In [1]:
import pandas as pd
import numpy as np
import gdown
import pyarrow.parquet as pq

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

Question 1: Dummies for Month and Week-of-Month

What is the ABSOLUTE CORRELATION VALUE of the most correlated dummy variable _w<week_of_month> with the binary outcome is_positive_growth_30d_future?

From the correlation analysis and modeling, you may have observed that October and November are potentially important seasonal months. In this task, you'll go further by generating dummy variables for both the Month and Week-of-Month (starting from 1). For example, the first week of October should be coded as: 'October_w1'.

Once you've generated these new variables, identify the one with the highest absolute correlation with is_positive_growth_30d_future, and round the result to three decimal places.

Suggested Steps

Use this StackOverflow reference to compute the week of the month using the following formula:
  * (d.day - 1) // 7 + 1

Create a new string variable that combines the month name and week of the month. Example: 'October_w1', 'November_w2', etc.

Add the new variable (e.g., month_wom) to your set of categorical features.

Your updated categorical feature list should include:

  * 'Month'
  * 'Weekday'
  * 'Ticker'
  * 'ticker_type'
  * 'month_wom'
  
Use pandas.get_dummies() to generate dummy variables for all categorical features.

  * This should result in approximately 115 dummy variables, including around 60 for the month_wom feature (12 months × up to 5 weeks).

Use DataFrame.corr() to compute the correlation between each feature and the target variable is_positive_growth_30d_future.

Filter the correlation results to include only the dummy variables generated from month_wom.

Create a new column named abs_corr in the correlation results that stores the absolute value of the correlations.

Sort the correlation results by abs_corr in descending order.

Identify and report the highest absolute correlation value among the month_wom dummy variables, rounded to three decimal places.

NOTE: new dummies will be used as features in the next tasks, please leave them in the dataset.

In [ ]:
#download data
file_id = "1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)

In [3]:
df = pq.read_table("data.parquet").to_pandas()

In [4]:
df.info()

<class 'pandas.DataFrame'>
Index: 240759 entries, 0 to 6018
Columns: 203 entries, Open to growth_btc_usd_365d
dtypes: datetime64[ns](3), float64(129), int32(64), int64(5), str(2)
memory usage: 317.7 MB


In [5]:
df[df.duplicated(subset=["Ticker", "Date"], keep="first")]

,Open,High,Low,Close_x,Volume,Dividends,Stock Splits,Ticker,Year,Month,...,growth_brent_oil_7d,growth_brent_oil_30d,growth_brent_oil_90d,growth_brent_oil_365d,growth_btc_usd_1d,growth_btc_usd_3d,growth_btc_usd_7d,growth_btc_usd_30d,growth_btc_usd_90d,growth_btc_usd_365d


In [6]:
print(f"Total rows: {len(df)}")
print(f"Unique (Ticker, Date) pairs: {df[['Ticker','Date']].drop_duplicates().shape[0]}")

Total rows: 240759
Unique (Ticker, Date) pairs: 240759


In [7]:
#set up numerical df columns

#growth indicators (but not future growth)
GROWTH = [g for g in df.keys() if (g.find("growth_") == 0) & (g.find("future") < 0)]
#leaving only Volume ==> generate ln(Volume)
OHLCV = ["Open", "High", "Low", "Close", "Adj Close_x", "Volume"]
#a small hack to avoid "divide by zero encountered in log"
df["ln_volume"] = df.Volume.replace(0, np.nan).fillna(1e-9).apply(lambda x: np.log(x))
#manually defined features
CUSTOM_NUMERICAL = [
    "SMA10", "SMA20", "growing_moving_average", "high_minus_low_relative","volatility", "ln_volume"
]
#All Supported Ta-lib indicators: https://github.com/TA-Lib/ta-lib-python/blob/master/docs/funcs.md
TECHNICAL_INDICATORS = [
    "adx", "adxr", "apo", "aroon_1", "aroon_2", "aroonosc", "bop", "cci", "cmo","dx", "macd",
    "macdsignal", "macdhist", "macd_ext", "macdsignal_ext", "macdhist_ext", "macd_fix",
    "macdsignal_fix", "macdhist_fix", "mfi", "minus_di", "mom", "plus_di", "dm", "ppo", "roc",
    "rocp", "rocr", "rocr100", "rsi", "slowk", "slowd", "fastk", "fastd", "fastk_rsi",
    "fastd_rsi", "trix", "ultosc", "willr", "ad", "adosc", "obv", "atr", "natr", "ht_dcperiod",
    "ht_dcphase", "ht_phasor_inphase", "ht_phasor_quadrature", "ht_sine_sine",
    "ht_sine_leadsine", "ht_trendmod", "avgprice", "medprice", "typprice", "wclprice"
]

TECHNICAL_PATTERNS = [g for g in df.keys() if g.find("cdl")>=0]

MACRO = [
    "gdppot_us_yoy", "gdppot_us_qoq", "cpi_core_yoy", "cpi_core_mom", "FEDFUNDS",
    "DGS1", "DGS5", "DGS10"
]

In [8]:
TO_PREDICT = [g for g in df.keys() if (g.find("future")>=0)]

In [9]:
NUMERICAL = GROWTH + TECHNICAL_INDICATORS + TECHNICAL_PATTERNS + CUSTOM_NUMERICAL + MACRO

In [10]:
#set up data to perform calculations
DATE_FILTER = "2000-01-01"
filtered_df = df[df.Date >= DATE_FILTER].copy()
filtered_df.info()

<class 'pandas.DataFrame'>
Index: 202292 entries, 3490 to 6018
Columns: 204 entries, Open to ln_volume
dtypes: datetime64[ns](3), float64(130), int32(64), int64(5), str(2)
memory usage: 268.6 MB


In [11]:
#get wom from Date column
filtered_df["wom"] = filtered_df["Date"].dt.day.apply(
    lambda day : str((day - 1) // 7 + 1)
)

filtered_df.wom.head()

3490    1
3491    1
3492    1
3493    1
3494    1
Name: wom, dtype: str

In [12]:
#use month name from Month
filtered_df["Month"] = filtered_df["Month"].dt.month_name()
filtered_df["Month"] = filtered_df["Month"].str.lower()

filtered_df.Month.head()

3490    january
3491    january
3492    january
3493    january
3494    january
Name: Month, dtype: str

In [13]:
#define new feature month_wom
filtered_df["month_wom"] = filtered_df["Month"] + "_w" + filtered_df["wom"]

filtered_df["month_wom"].head()

3490    january_w1
3491    january_w1
3492    january_w1
3493    january_w1
3494    january_w1
Name: month_wom, dtype: str

In [14]:
#remove aux column wom
filtered_df = filtered_df.drop(columns=["wom"])

In [15]:
#define the list of updated categorical features
CAT = ["Month", "Weekday", "Ticker", "ticker_type", "month_wom"]

In [16]:
#review dtypes
filtered_df[CAT].info()

<class 'pandas.DataFrame'>
Index: 202292 entries, 3490 to 6018
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   Month        202292 non-null  str  
 1   Weekday      202292 non-null  int32
 2   Ticker       202292 non-null  str  
 3   ticker_type  202292 non-null  str  
 4   month_wom    202292 non-null  str  
dtypes: int32(1), str(4)
memory usage: 13.1 MB


In [17]:
#Weekday as str dtype
filtered_df["Weekday"] = filtered_df["Weekday"].astype(str)

In [18]:
#generate dummy variables for selected categorical features
dummy_df = pd.get_dummies(filtered_df[CAT], dtype="int32")

In [19]:
#check 115 dummy vars
dummy_df.shape[1]

115

In [20]:
#check 60 vars for month_wom feature
dummy_df.columns.str.contains("month_wom").sum()

np.int64(60)

In [21]:
#concatenate filtered and dummy dfs
df_dummies = pd.concat([filtered_df, dummy_df], axis=1)

df_dummies.head()

,Open,High,Low,Close_x,Volume,Dividends,Stock Splits,Ticker,Year,Month,...,month_wom_october_w1,month_wom_october_w2,month_wom_october_w3,month_wom_october_w4,month_wom_october_w5,month_wom_september_w1,month_wom_september_w2,month_wom_september_w3,month_wom_september_w4,month_wom_september_w5
3490,35.623602,36.002980,33.992277,35.377007,53228400.0,0.0,0.0,MSFT,2000,january,...,0,0,0,0,0,0,0,0,0,0
3491,34.466525,35.547754,34.068178,34.181992,54119000.0,0.0,0.0,MSFT,2000,january,...,0,0,0,0,0,0,0,0,0,0
3492,33.726722,35.320110,33.195592,34.542385,64059600.0,0.0,0.0,MSFT,2000,january,...,0,0,0,0,0,0,0,0,0,0
3493,34.049185,34.561345,32.892081,33.385273,54976600.0,0.0,0.0,MSFT,2000,january,...,0,0,0,0,0,0,0,0,0,0
3494,32.967948,34.068144,32.569601,33.821548,62013600.0,0.0,0.0,MSFT,2000,january,...,0,0,0,0,0,0,0,0,0,0


In [22]:
#select features for correlation
TARGET = "is_positive_growth_30d_future"
DUMMIES = dummy_df.keys().to_list()
CORR_COLS = NUMERICAL + DUMMIES + TO_PREDICT

In [23]:

df_dummies[NUMERICAL+DUMMIES].info()

<class 'pandas.DataFrame'>
Index: 202292 entries, 3490 to 6018
Columns: 299 entries, growth_1d to month_wom_september_w5
dtypes: float64(121), int32(177), int64(1)
memory usage: 326.4 MB


In [24]:
#compute abs correlation between each feature and is_positive_growth_30d_future -> ipg_30d_fut
corr_ipg_30d_fut = df_dummies[CORR_COLS].corr().abs().sort_values(TARGET, ascending=False)
corr_ipg_30d_fut

,growth_1d,growth_3d,growth_7d,growth_30d,growth_90d,growth_365d,growth_dax_1d,growth_dax_3d,growth_dax_7d,growth_dax_30d,...,month_wom_october_w3,month_wom_october_w4,month_wom_october_w5,month_wom_september_w1,month_wom_september_w2,month_wom_september_w3,month_wom_september_w4,month_wom_september_w5,growth_future_30d,is_positive_growth_30d_future
is_positive_growth_30d_future,0.012003,0.014807,0.015087,0.018983,0.010837,0.020835,0.004278,0.010479,0.015922,0.018415,...,0.017856,0.024584,0.012711,0.014046,0.012802,0.009193,0.014618,0.013410,0.698218,1.000000
growth_future_30d,0.023421,0.029279,0.034818,0.034438,0.009235,0.034213,0.002127,0.004536,0.011155,0.022994,...,0.024150,0.029675,0.014888,0.011239,0.013496,0.004477,0.020099,0.014172,1.000000,0.698218
DGS10,0.002028,0.005100,0.010206,0.018398,0.010576,0.063619,0.000495,0.004594,0.006873,0.007875,...,0.002972,0.003056,0.005425,0.008148,0.008797,0.007008,0.014270,0.016206,0.046501,0.071102
DGS5,0.002592,0.005732,0.010297,0.016203,0.011713,0.085370,0.000567,0.004895,0.005040,0.001445,...,0.005535,0.005759,0.002134,0.004865,0.005883,0.006139,0.011989,0.016321,0.046799,0.064277
growth_brent_oil_365d,0.012718,0.019822,0.025880,0.044373,0.062798,0.176707,0.018043,0.033835,0.048825,0.090174,...,0.007030,0.002491,0.021913,0.014687,0.020368,0.015173,0.000194,0.010712,0.075041,0.058880
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Weekday_0,0.008408,0.007380,0.004072,0.001302,0.002048,0.000626,0.002303,0.013661,0.009407,0.005117,...,0.002276,0.003855,0.001202,0.035788,0.004036,0.000570,0.002416,0.005774,0.001558,0.000241
month_wom_march_w1,0.004022,0.008222,0.026190,0.016021,0.008812,0.004716,0.011922,0.011798,0.034817,0.021346,...,0.019775,0.019640,0.012866,0.018920,0.020046,0.020118,0.019848,0.010447,0.001028,0.000171
cdl3starsinsouth,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cdlconcealbabyswall,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
#select target variable to apply month_wom filter
corr_ipg_30d_fut_df = corr_ipg_30d_fut[TARGET].copy()

In [26]:
#use index as column
corr_ipg_30d_fut_df = corr_ipg_30d_fut_df.reset_index()
#rename columns, use abs_corr to save corr value
corr_ipg_30d_fut_df.columns = ["dummy_feature", "abs_corr"]
#filter month_wom features
month_wom_filter = corr_ipg_30d_fut_df[corr_ipg_30d_fut_df["dummy_feature"].str.contains("month_wom")]
#round to 3 decimal places
month_wom_filter["abs_corr"] = month_wom_filter["abs_corr"].round(3)
month_wom_filter.head()

,dummy_feature,abs_corr
43,month_wom_october_w4,0.025
45,month_wom_november_w3,0.023
49,month_wom_february_w1,0.021
57,month_wom_january_w2,0.020
58,month_wom_january_w5,0.020


Question 2: Define New "Hand" Rules on Macro and Technical Indicator Variables

What is the precision score for the best of the NEW predictions (pred3 or pred4), rounded to 3 digits after the comma?

In this task, you'll apply insights from the visualized decision tree (clf10) (see Code Snippet 5: 1.4.4 Visualisation) to manually define and evaluate new predictive rules.

Define two new 'hand' rules based on branches that lead to 'positive' predictions in the tree:

  * pred3_manual_dgs10_5: (DGS10 <= 4.5) & (DGS5 <= 4)
  * pred4_manual_dgs10_fedfunds: (DGS10 > 4) & (FEDFUNDS <= 4.795)

Hint: This is not exactly the same condition as in the estimated tree (original: (cpi_core_yoy<=0.023) & (DGS10 <= 1.745); (cpi_core_yoy>0.023) & (FEDFUNDS <= 4.96)), since the Decision Tree is quite different in 2026 vs. 2025.

Extend Code Snippet 3 (Manual "hand rule" predictions):

  * Implement and apply the above two rules (pred3, pred4) to your dataset.
  * Add the resulting predictions as new columns in your dataframe (e.g., new_df).

Compute precision:

  * For the rule that does make positive predictions on the TEST set, compute its precision score.
  * Use standard precision metrics (TP / (TP + FP)).
  * Round the precision score to three decimal places. Example: If your result is 0.57897, your final answer should be: 0.579.

Hint: This should already be visible in the code output, as the IS_CORRECT and PREDICTIONS sets should automatically include the new columns.

In [ ]:
#code from module 3 notebook
def temporal_split(df, min_date, max_date, train=0.7, val=0.15, test=0.15):
    """
    Splits a DataFrame into three buckets based on the temporal order of the "Date" column

    Args:
        df (DataFrame): The DataFrame to split
        min_date (str or Timestamp): Minimum date in the DataFrame
        max_date (str or Timestamp): Maximum date in the DataFrame
        train_prop (float): Proportion of data for training set (default: 0.7)
        val_prop (float): Proportion of data for validation set (default: 0.15)
        test_prop (float): Proportion of data for test set (default: 0.15)

    Returns:
        DataFrame: The input DataFrame with a new column "split" indicating the split for each row
    """
    train_end = min_date + pd.Timedelta(days=(max_date - min_date).days * train)
    val_end = train_end + pd.Timedelta(days=(max_date - min_date).days * val)

    split_labels = []
    for date in df["Date"]:
        if date <= train_end:
            split_labels.append("train")
        elif date <= val_end:
            split_labels.append("validation")
        else:
            split_labels.append("test")

    df["split"] = split_labels

    return df

In [28]:
#get date tresholds
min_date = df_dummies.Date.min()
max_date = df_dummies.Date.max()
#get split
corr_df = temporal_split(df_dummies, min_date=min_date, max_date=max_date)
#check split proportion
corr_df["split"].value_counts() / len(corr_df)

split
train         0.676458
test          0.163936
validation    0.159606
Name: count, dtype: float64

In [29]:
corr_df.groupby(["split"])["Date"].agg({"min","max","count"})

,min,max,count
split,,,
test,2022-09-16,2026-09-18,33163
train,2000-01-03,2018-09-13,136842
validation,2018-09-14,2022-09-15,32287


In [30]:
#generate manual predictions
corr_df["pred0_manual_cci"] = (corr_df.cci > 200).astype(int)
corr_df["pred1_manual_prev_g1"] = (corr_df.growth_30d > 1).astype(int)
corr_df["pred2_manual_prev_g1_and_snp"] = (
    (corr_df["growth_30d"] > 1) & (corr_df["growth_snp500_30d"] > 1)
).astype(int)

In [31]:
#generate pred3_manual_dgs10_5: (DGS10 <= 4.5) & (DGS5 <= 4)
corr_df["pred3_manual_dgs10_5"] = (
    (corr_df["DGS10"] <= 4.5) & (corr_df["DGS5"] <= 4)
).astype(int)

In [32]:
#generate pred4_manual_dgs10_fedfunds: (DGS10 > 4) & (FEDFUNDS <= 4.795)
corr_df["pred4_manual_dgs10_fedfunds"] = (
    (corr_df["DGS10"] > 4) & (corr_df["FEDFUNDS"] <= 4.795)
).astype(int)

In [33]:
corr_df.groupby(by="split")[TARGET].describe()

,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,33163.0,0.570093,0.495070,0.0,0.0,1.0,1.0,1.0
train,136842.0,0.591449,0.491568,0.0,0.0,1.0,1.0,1.0
validation,32287.0,0.601449,0.489607,0.0,0.0,1.0,1.0,1.0


In [34]:
#list of prediction names
PREDICTIONS = [pred_name for pred_name in corr_df.keys() if pred_name.startswith("pred")]
PREDICTIONS

['pred0_manual_cci',
 'pred1_manual_prev_g1',
 'pred2_manual_prev_g1_and_snp',
 'pred3_manual_dgs10_5',
 'pred4_manual_dgs10_fedfunds']

In [35]:
#create column is_correct from each prediction
for prediction in PREDICTIONS:
    prefix = prediction.split("_")[0]
    corr_df[f"is_correct_{prefix}"] = (
        corr_df[prediction] == corr_df[TARGET]
    ).astype(int)

In [36]:
corr_df[PREDICTIONS].describe()

,pred0_manual_cci,pred1_manual_prev_g1,pred2_manual_prev_g1_and_snp,pred3_manual_dgs10_5,pred4_manual_dgs10_fedfunds
count,202292.000000,202292.000000,202292.000000,202292.00000,202292.000000
mean,0.026551,0.591314,0.455708,0.75349,0.230024
std,0.160767,0.491592,0.498036,0.43098,0.420849
min,0.000000,0.000000,0.000000,0.00000,0.000000
25%,0.000000,0.000000,0.000000,1.00000,0.000000
50%,0.000000,1.000000,0.000000,1.00000,0.000000
75%,0.000000,1.000000,1.000000,1.00000,0.000000
max,1.000000,1.000000,1.000000,1.00000,1.000000


In [37]:
#list of is_correct names
IS_CORRECT = [is_correct_name for is_correct_name in corr_df.keys() if is_correct_name.startswith("is_correct")]
IS_CORRECT

['is_correct_pred0',
 'is_correct_pred1',
 'is_correct_pred2',
 'is_correct_pred3',
 'is_correct_pred4']

In [38]:
corr_df[IS_CORRECT].describe()

,is_correct_pred0,is_correct_pred1,is_correct_pred2,is_correct_pred3,is_correct_pred4
count,202292.000000,202292.000000,202292.000000,202292.000000,202292.000000
mean,0.414683,0.506664,0.483282,0.560887,0.426670
std,0.492668,0.499957,0.499722,0.496280,0.494595
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,1.000000,0.000000,1.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000


In [39]:
#code from module 3 notebook
for idx, col in enumerate(IS_CORRECT):
  #define precision for predictions on a test dataset
  prediction_col = PREDICTIONS[idx]
  is_correct_col = col
  filter = ((corr_df.split == "test") & (corr_df[prediction_col] == 1))

  print(f"prediction_col:{prediction_col}, is_correct_col:{is_correct_col}")
  print(corr_df[filter][is_correct_col].value_counts())
  print((corr_df[filter][is_correct_col].value_counts() / len(corr_df[filter])).round(3))

  print("---------")

prediction_col:pred0_manual_cci, is_correct_col:is_correct_pred0
is_correct_pred0
1    520
0    401
Name: count, dtype: int64
is_correct_pred0
1    0.565
0    0.435
Name: count, dtype: float64
---------
prediction_col:pred1_manual_prev_g1, is_correct_col:is_correct_pred1
is_correct_pred1
1    11086
0     8072
Name: count, dtype: int64
is_correct_pred1
1    0.579
0    0.421
Name: count, dtype: float64
---------
prediction_col:pred2_manual_prev_g1_and_snp, is_correct_col:is_correct_pred2
is_correct_pred2
1    8728
0    6516
Name: count, dtype: int64
is_correct_pred2
1    0.573
0    0.427
Name: count, dtype: float64
---------
prediction_col:pred3_manual_dgs10_5, is_correct_col:is_correct_pred3
is_correct_pred3
1    9351
0    6559
Name: count, dtype: int64
is_correct_pred3
1    0.588
0    0.412
Name: count, dtype: float64
---------
prediction_col:pred4_manual_dgs10_fedfunds, is_correct_col:is_correct_pred4
is_correct_pred4
1    8001
0    7920
Name: count, dtype: int64
is_correct_pred4
1   

In [40]:
#precision for pred3 and pred4
for col in PREDICTIONS[-2:]:
    target = corr_df[corr_df["split"] == "test"][TARGET]
    pred = corr_df[corr_df["split"] == "test"][col]

    tp = ((pred == 1) & (pred == target)).sum()
    fp = ((pred == 1) & (pred != target)).sum()

    #using tp, fp
    precision = tp / (tp + fp)

    print(f"\nprediction {col} \nprecision (tp, fp): {np.round(precision, 3)}")
    #from precision_score
    print("prec_score:", np.round(precision_score(target, pred), 3))


prediction pred3_manual_dgs10_5 
precision (tp, fp): 0.588
prec_score: 0.588

prediction pred4_manual_dgs10_fedfunds 
precision (tp, fp): 0.503
prec_score: 0.503


Question 3: Unique Correct Predictions from a 10-Level Decision Tree Classifier (pred5_clf_10)

What is the total number of records in the TEST dataset where the new prediction pred5_clf_10 is correct, while all 'hand' rule predictions (pred0 to pred4) are incorrect?

To ensure reproducibility, please include the following parameter in the Decision Tree Classifier:

```python
    clf = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
```

Step 1: Train the Decision Tree and Generate Predictions

  * Initialize a Decision Tree Classifier with a maximum depth of 10 and set random_state=42 for reproducibility.
  * Fit the classifier on the combined TRAIN and VALIDATION datasets.
  * Use the trained model to predict on the entire dataset (TRAIN + VALIDATION + TEST).
  * Store these predictions in a new column named pred5_clf_10 within your main dataframe.
  * Hint: When predicting on the entire dataset, it's easy to join the predictions with the full DataFrame, since the number of records and their order remain the same. You will need to define X_all and y_all and apply the same cleaning steps used previously for X_train, y_train, X_test, and y_test. This makes it straightforward to define a new column, for example:
  ```python
    df['pred5_clf_10'] = <predictions vector from clf10.predict(X_all)>
  ```
 
Step 2: Identify Unique Correct Predictions by pred5_clf_10

  * Create a new boolean column, only_pred5_is_correct, that is True only when:
    * The prediction from pred5_clf_10 is correct (i.e., matches the true label).
    * All other hand rule predictions (pred0 through pred4) are incorrect.

Step 3: Count Unique Correct Predictions on the TEST Set
  * Convert the only_pred5_is_correct column from boolean to integer.
  * Filter the dataframe for records belonging to the TEST dataset.
  * Count how many records in the TEST set have only_pred5_is_correct equal to 1.
  * Report this count as your final answer.

Advanced (Optional)
  * To generalize this for many prediction columns (e.g., pred0 to pred5), define a function that can be applied to an entire dataframe row.
  * This function should identify whether a specific prediction (predX) is uniquely correct (correct while all others are incorrect).
  * This approach avoids hardcoding conditions for each predictor and scales easily.
  * For examples of how to apply functions to rows in pandas, see this helpful resource:
    [Pandas apply function to every row](https://medium.com/@amit25173/tips-for-the-pandas-apply-function-to-every-row-e15929325985)

In [41]:
#split data
#already set just placing it here for clear reference
TARGET = "is_positive_growth_30d_future"

#filters for train and test data
train_mask = corr_df.split.isin(["train","validation"])
test_mask = corr_df.split.isin(["test"])

#train, test splits
train = corr_df[train_mask].copy()
test = corr_df[test_mask].copy()
all_splits = corr_df.copy()

#additional features
MERGING = [TARGET, "Date", "Ticker"]
feature_list = NUMERICAL + DUMMIES + MERGING

x_train = train[feature_list].copy()
x_test = test[feature_list].copy()
x_all = all_splits[feature_list].copy()

print(f"x_train:{x_train.shape}, x_test:{x_test.shape}, x_all:{x_all.shape}")

x_train:(169129, 302), x_test:(33163, 302), x_all:(202292, 302)


Note: The difference in column count is because of the addition of Month and Week of Month to dummy variables from Q1

In [42]:
#replace +,- np.inf and NaN with zero
inf_values = [np.inf, -np.inf]
#inf for nan
x_train = x_train.replace(inf_values, np.nan)
x_test = x_test.replace(inf_values, np.nan)
x_all = x_all.replace(inf_values, np.nan)
#nan for 0
x_train = x_train.fillna(0)
x_test = x_test.fillna(0)
x_all = x_all.fillna(0)

print(f"x_train (missing): {x_train.isna().sum().sum()}")
print(f"x_test (missing): {x_test.isna().sum().sum()}")
print(f"x_all (missing): {x_all.isna().sum().sum()}")

print(f"x_train:{x_train.shape}, x_test:{x_test.shape}, x_all:{x_all.shape}")

x_train (missing): 0
x_test (missing): 0
x_all (missing): 0
x_train:(169129, 302), x_test:(33163, 302), x_all:(202292, 302)


In [43]:
#isolate target
y_train = x_train[TARGET]
y_test = x_test[TARGET]
y_all = x_all[TARGET]

#remove target from train, test
del x_train[TARGET]
del x_test[TARGET]
del x_all[TARGET]

In [44]:
#set up decision tree classifier
dt_clf = DecisionTreeClassifier(max_depth=10, random_state=42)
#fit the decision tree classifier, remove Date and Ticker
dt_clf.fit(x_train.drop(["Date", "Ticker"], axis=1), y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples i

In [45]:
#predict using df_predict and save to main df as pred5_clf_10
corr_df["pred5_clf_10"] = dt_clf.predict(x_all.drop(["Date", "Ticker"], axis=1))

In [46]:
#identify unique correct predictions for pred5_clf_10
#list of prediction names for decision tree classifier
PREDICTIONS_DTC = [pred_name for pred_name in corr_df.keys() if pred_name.startswith("pred")]
PREDICTIONS_DTC

['pred0_manual_cci',
 'pred1_manual_prev_g1',
 'pred2_manual_prev_g1_and_snp',
 'pred3_manual_dgs10_5',
 'pred4_manual_dgs10_fedfunds',
 'pred5_clf_10']

In [47]:
#create column is_correct for prediction if required
for prediction in PREDICTIONS_DTC:
    prefix = prediction.split("_")[0]
    pred_name = "is_correct_" + prefix
    
    if pred_name not in corr_df.columns.to_list():
        print("adding:", pred_name)
        corr_df[pred_name] = (
            corr_df[prediction] == corr_df[TARGET]
        ).astype(int)

adding: is_correct_pred5


In [48]:
PREDICTIONS_DTC[:5]

['pred0_manual_cci',
 'pred1_manual_prev_g1',
 'pred2_manual_prev_g1_and_snp',
 'pred3_manual_dgs10_5',
 'pred4_manual_dgs10_fedfunds']

In [49]:
PREDICTIONS_DTC[-1]

'pred5_clf_10'

In [50]:
#create only_pred5_is_correct, that is True only when:
    #The prediction from pred5_clf_10 is correct.
    #All other hand rule predictions (pred0 through pred4) are incorrect.

#get column names
pred_col = PREDICTIONS_DTC[-1]
other = PREDICTIONS_DTC[:5]
#apply condition filter
true_pred5 = (corr_df[pred_col] == 1) & (corr_df[other] == 0).all(axis=1)
#define new column with boolean series result
corr_df["only_pred5_is_correct"] = true_pred5

In [51]:
#count unique correct predictions on the test set
true_pred5_count_mask = (corr_df["split"] == "test") & (corr_df["only_pred5_is_correct"] == 1)
correct_pred = corr_df[true_pred5_count_mask]["only_pred5_is_correct"]
correct_pred.shape

(1890,)

Question 4: Hyperparameter tuning for a Decision Tree

What is the optimal tree depth (from 1 to 12) for a DecisionTreeClassifier?

NOTE: please include random_state=42 to the Decision Tree Classifier initialization (e.g., clf = DecisionTreeClassifier(max_depth=max_depth, random_state=42)) to ensure consistency in results.

Instructions:

  * Iterate through max_depth values from 1 to 12.
  * For each max_depth:
    * Train a Decision Tree Classifier with the current max_depth on the combined TRAIN+VALIDATION dataset.
  * Optionally, visualize how the 'head' (top levels) of each fitted tree changes with increasing tree depth. You can use:
    * sklearn.tree.plot_tree() for graphical visualization, or
    * The compact textual approach with export_text() function. For example:
    ```python
    from sklearn.tree import export_text
    tree_rules = export_text(model, feature_names=list(X_train), max_depth=3)
    print(tree_rules)
    ```

  * Calculate the precision score on the TEST dataset for each fitted tree. You may also track precision on the VALIDATION dataset to observe signs of overfitting.
  * Identify the optimal max_depth where the precision score on the TEST dataset is highest. This value is your best_max_depth.
  * Using best_max_depth, retrain the Decision Tree Classifier on the combined TRAIN+VALIDATION set.
  * Predict on the entire dataset (TRAIN + VALIDATION + TEST) and add the predictions as a new column pred6_clf_best in your dataframe new_df.
  * Compare the precision score of the tuned tree with previous predictions (pred0 to pred5). You should observe an improvement, ideally achieving precision > 0.58, indicating the tuned tree outperforms earlier models.

Advanced (Optional)

  * Plot the precision (or accuracy) scores against the max_depth values to detect saturation or overfitting trends.
  * Observe the trade-off between model complexity (deeper trees) and generalization capability.
  * For more information, consult the [scikit-learn Decision Trees documentation](https://scikit-learn.org/stable/modules/tree.html).

In [52]:
#set value to try
max_depth = 12
best_depth = 0
prec_score = 0.0

for depth in range(1, max_depth + 1):
    dtc = DecisionTreeClassifier(max_depth=depth, random_state=42)
    #fit dtc on tran and validation splits
    dtc.fit(x_train.drop(["Date", "Ticker"], axis=1), y_train)
    #predict on test split
    y_pred = dtc.predict(x_test.drop(["Date", "Ticker"], axis=1))
    #precision score on test split
    prec_score_test = precision_score(y_test, y_pred)
    #identify best depth
    if prec_score_test > prec_score:
        prec_score = prec_score_test
        best_depth = depth
        print(f"depth: {best_depth}, prec_score: {prec_score}")

print(f"Optimal depth: {best_depth}")

depth: 1, prec_score: 0.5700931761300244
depth: 2, prec_score: 0.6225340136054421
depth: 4, prec_score: 0.6291747790924068
Optimal depth: 4


In [53]:
#train using found depth
dtc = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
dtc.fit(x_train.drop(["Date", "Ticker"], axis=1), y_train)
#add predictions as new column
corr_df["pred6_clf_best"] = dtc.predict(x_all.drop(["Date", "Ticker"], axis=1))

In [54]:
#list of prediction names for decision tree classifier
PREDICTIONS_DTC = [pred_name for pred_name in corr_df.keys() if pred_name.startswith("pred")]
PREDICTIONS_DTC

['pred0_manual_cci',
 'pred1_manual_prev_g1',
 'pred2_manual_prev_g1_and_snp',
 'pred3_manual_dgs10_5',
 'pred4_manual_dgs10_fedfunds',
 'pred5_clf_10',
 'pred6_clf_best']

In [55]:
#create column is_correct for prediction if required
for prediction in PREDICTIONS_DTC:
    prefix = prediction.split("_")[0]
    pred_name = "is_correct_" + prefix
    
    if pred_name not in corr_df.columns.to_list():
        print("adding:", pred_name)
        corr_df[pred_name] = (
            corr_df[prediction] == corr_df[TARGET]
        ).astype(int)

adding: is_correct_pred6


In [56]:
#list of is_correct names
IS_CORRECT = [is_correct_name for is_correct_name in corr_df.keys() if is_correct_name.startswith("is_correct")]
IS_CORRECT

['is_correct_pred0',
 'is_correct_pred1',
 'is_correct_pred2',
 'is_correct_pred3',
 'is_correct_pred4',
 'is_correct_pred5',
 'is_correct_pred6']

In [57]:
#code from module 3 notebook, i used it to compare the results of pred_0 - pred_5
for idx, col in enumerate(IS_CORRECT):
  #define precision for predictions on a test dataset
  prediction_col = PREDICTIONS_DTC[idx]
  is_correct_col = col
  filter = ((corr_df.split == "test") & (corr_df[prediction_col] == 1))

  print(f"prediction_col:{prediction_col}, is_correct_col:{is_correct_col}")
  #print(corr_df[filter][is_correct_col].value_counts())
  print((corr_df[filter][is_correct_col].value_counts() / len(corr_df[filter])).round(3))

  print("---------")

prediction_col:pred0_manual_cci, is_correct_col:is_correct_pred0
is_correct_pred0
1    0.565
0    0.435
Name: count, dtype: float64
---------
prediction_col:pred1_manual_prev_g1, is_correct_col:is_correct_pred1
is_correct_pred1
1    0.579
0    0.421
Name: count, dtype: float64
---------
prediction_col:pred2_manual_prev_g1_and_snp, is_correct_col:is_correct_pred2
is_correct_pred2
1    0.573
0    0.427
Name: count, dtype: float64
---------
prediction_col:pred3_manual_dgs10_5, is_correct_col:is_correct_pred3
is_correct_pred3
1    0.588
0    0.412
Name: count, dtype: float64
---------
prediction_col:pred4_manual_dgs10_fedfunds, is_correct_col:is_correct_pred4
is_correct_pred4
1    0.503
0    0.497
Name: count, dtype: float64
---------
prediction_col:pred5_clf_10, is_correct_col:is_correct_pred5
is_correct_pred5
1    0.599
0    0.401
Name: count, dtype: float64
---------
prediction_col:pred6_clf_best, is_correct_col:is_correct_pred6
is_correct_pred6
1    0.629
0    0.371
Name: count, dtype:

[EXPLORATORY] Question 5: What data is missing?

Now that you have gained insights from the correlation analysis and Decision Tree results regarding the most influential variables, suggest new indicators you would like to include in the dataset and explain your reasoning.

Alternatively, you may propose a completely different approach based on your intuition, provided it remains relevant to the shared dataset of the largest stocks from India, the EU, and the US. If you choose this route, please also specify the data source.

I would like to introduce the indicators used in the previous module considering the 21 trading days used for growth but this time use a different model and technique to select most important features.